# DNN Training Pipeline with 5-Fold Cross-Validation

This notebook trains a clinical feature-based DNN for Alzheimer's classification using:
- **Data**: 5-fold stratified cross-validation
- **Features**: 18 clinical features (14 numeric + 4 categorical)
- **Imbalance Handling**: SMOTE oversampling on training folds only
- **Architecture**: AlzheimerDNN with skip connections (18 features → 128-d embedding → 4 classes)
- **Optimization**: AdamW + CosineAnnealingLR scheduler
- **Output**: Feature importance analysis and model checkpoint

## 1. Setup: Imports and Configuration

In [1]:
import sys
from pathlib import Path

# Add parent directory to path so we can import models
sys.path.insert(0, str(Path(".").resolve().parent))

from __future__ import annotations

import copy
import importlib
import json
import random
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from imblearn.over_sampling import SMOTE

import models.dnn_tabular as dnn_tabular_module
importlib.reload(dnn_tabular_module)
from models.dnn_tabular import AlzheimerDNN, FeatureImportanceAnalyzer, FEATURE_NAMES

# Set matplotlib to display plots inline
%matplotlib inline

print("Imports successful.")

Imports successful.


## 2. Helper Functions

## 1.5 Configuration

In [26]:
# Configuration
import os

SEED = 42
NUM_CLASSES = 4
N_SPLITS = 5
EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
T_MAX = 30  # For CosineAnnealingLR

# Paths
# Allow override via environment variable 'ALZHEIMER_DATA_PATH'
BASE_DIR = Path.cwd().parent
DATA_PATH = Path(os.environ.get('ALZHEIMER_DATA_PATH', BASE_DIR / 'data/raw/clinical/alzheimers_dataset.csv'))
if not DATA_PATH.exists():
    candidates = list(BASE_DIR.rglob('alzheimers_dataset.csv'))
    if candidates:
        DATA_PATH = candidates[0]
        print(f'Using discovered dataset at: {DATA_PATH}')
    else:
        raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

SAVE_PATH = Path('models/saved/dnn_best.pth')
PLOT_PATH = Path('evaluation/plots/dnn_feature_importance.png')
TARGET_COLUMN = 'dx1'

# Engineer 18 numeric features from the available clinical columns.
ENGINEERED_FEATURE_NAMES = [
    'age',
    'mmse',
    'cdr',
    'memory',
    'gender_encoded',
    'age_x_mmse',
    'age_x_cdr',
    'age_x_memory',
    'mmse_x_cdr',
    'mmse_x_memory',
    'cdr_x_memory',
    'age_sq',
    'mmse_sq',
    'cdr_sq',
    'memory_sq',
    'age_minus_mmse',
    'age_plus_memory',
    'age_over_mmse',
]

NUMERIC_FEATURES = ENGINEERED_FEATURE_NAMES.copy()
CATEGORICAL_FEATURES = []
FEATURE_NAMES = ENGINEERED_FEATURE_NAMES.copy()
CLASS_NAMES = []
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Configuration loaded.')
print(f"Device: {device}")
print(f"CUDA Available: {torch.cuda.is_available()}")

Configuration loaded.
Device: cpu
CUDA Available: False


In [27]:
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print("Seed set to:", SEED)

Seed set to: 42


In [28]:
class ClinicalTabularPreprocessor:
    """Impute missing values, encode categoricals, and scale numeric features."""

    def __init__(self, numeric_features: Sequence[str], categorical_features: Sequence[str]):
        self.numeric_features = list(numeric_features)
        self.categorical_features = list(categorical_features)
        self.feature_order = list(self.numeric_features) + list(self.categorical_features)
        self.numeric_medians: Dict[str, float] = {}
        self.categorical_modes: Dict[str, str] = {}
        self.label_encoders: Dict[str, LabelEncoder] = {}
        self.scaler = StandardScaler()
        self.is_fitted = False

    @staticmethod
    def _normalize_category(value) -> str:
        return str(value)

    def fit(self, frame: pd.DataFrame) -> "ClinicalTabularPreprocessor":
        working = frame.copy()

        for column in self.numeric_features:
            working[column] = pd.to_numeric(working[column], errors="coerce")
            self.numeric_medians[column] = float(working[column].median())
            working[column] = working[column].fillna(self.numeric_medians[column])

        for column in self.categorical_features:
            normalized = working[column].map(self._normalize_category)
            mode_value = normalized.mode(dropna=True).iloc[0]
            self.categorical_modes[column] = mode_value
            filled = normalized.fillna(mode_value)
            encoder = LabelEncoder()
            encoder.fit(filled)
            self.label_encoders[column] = encoder
            working[column] = encoder.transform(filled)

        self.scaler.fit(working[self.numeric_features].to_numpy(dtype=np.float32))
        self.is_fitted = True
        return self

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        if not self.is_fitted:
            raise RuntimeError("ClinicalTabularPreprocessor must be fitted before transform().")

        working = frame.copy()

        for column in self.numeric_features:
            working[column] = pd.to_numeric(working[column], errors="coerce")
            working[column] = working[column].fillna(self.numeric_medians[column])

        for column in self.categorical_features:
            normalized = working[column].map(self._normalize_category)
            filled = normalized.fillna(self.categorical_modes[column])
            working[column] = self.label_encoders[column].transform(filled)

        working[self.numeric_features] = self.scaler.transform(working[self.numeric_features].to_numpy(dtype=np.float32))
        return working[self.feature_order].to_numpy(dtype=np.float32)

    def fit_transform(self, frame: pd.DataFrame) -> np.ndarray:
        return self.fit(frame).transform(frame)

    def to_torch(self, x: np.ndarray, device: Optional[torch.device] = None) -> torch.Tensor:
        frame = pd.DataFrame(x, columns=self.feature_order)
        transformed = self.transform(frame)
        tensor = torch.tensor(transformed, dtype=torch.float32)
        return tensor.to(device) if device is not None else tensor

print("ClinicalTabularPreprocessor class defined.")

ClinicalTabularPreprocessor class defined.


In [29]:
def load_dataset() -> pd.DataFrame:
    if not DATA_PATH.exists():
        raise FileNotFoundError(f'Dataset not found: {DATA_PATH}')

    frame = pd.read_csv(DATA_PATH)
    required_columns = {'Subject', 'Gender', 'mmse', 'ageAtEntry', 'cdr', 'memory', TARGET_COLUMN}
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise ValueError(f'Missing required columns in dataset: {missing_columns}')

    return frame

print('load_dataset function defined.')

load_dataset function defined.


In [30]:
def make_loaders(features: np.ndarray, targets: np.ndarray, batch_size: int = BATCH_SIZE, shuffle: bool = True) -> DataLoader:
    feature_tensor = torch.tensor(features, dtype=torch.float32)
    target_tensor = torch.tensor(targets, dtype=torch.long)
    dataset = TensorDataset(feature_tensor, target_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

print("make_loaders function definedd.")

make_loaders function definedd.


In [32]:
def make_smote(x: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    class_counts = np.bincount(y, minlength=NUM_CLASSES)
    minority_count = int(class_counts[class_counts > 0].min()) if np.any(class_counts > 0) else 1
    k_neighbors = max(1, min(5, minority_count - 1))
    smote = SMOTE(random_state=SEED, k_neighbors=k_neighbors)
    return smote.fit_resample(x, y)

print("make_smote function defined.")

make_smote function defined.


In [33]:
def train_one_epoch(model, loader, criterion, optimizer, device) -> Tuple[float, float]:
    model.train()
    running_loss = 0.0
    all_targets: List[int] = []
    all_predictions: List[int] = []

    for features, targets in loader:
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(features)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)
        all_targets.extend(targets.detach().cpu().tolist())
        all_predictions.extend(logits.argmax(dim=1).detach().cpu().tolist())

    average_loss = running_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    return average_loss, accuracy

print("train_one_epoch function defined.")

train_one_epoch function defined.


In [34]:
def evaluate(model, loader, criterion, device) -> Tuple[float, float, np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    running_loss = 0.0
    all_targets: List[int] = []
    all_predictions: List[int] = []
    all_probabilities: List[np.ndarray] = []

    with torch.no_grad():
        for features, targets in loader:
            features = features.to(device)
            targets = targets.to(device)

            logits, _ = model(features)
            loss = criterion(logits, targets)
            probabilities = torch.softmax(logits, dim=1)

            running_loss += loss.item() * features.size(0)
            all_targets.extend(targets.detach().cpu().tolist())
            all_predictions.extend(logits.argmax(dim=1).detach().cpu().tolist())
            all_probabilities.append(probabilities.detach().cpu().numpy())

    average_loss = running_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    y_true = np.array(all_targets)
    y_pred = np.array(all_predictions)
    y_prob = np.concatenate(all_probabilities, axis=0)
    return average_loss, accuracy, y_true, y_pred, y_prob

print("evaluate function defined.")

evaluate function defined.


In [35]:
def save_checkpoint(payload: Dict) -> None:
    SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, SAVE_PATH)

print("save_checkpoint function defined.")

save_checkpoint function defined.


In [36]:
def save_feature_importance_plot(analyzer: FeatureImportanceAnalyzer, x: np.ndarray, y: np.ndarray) -> None:
    PLOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    analyzer.compute_importance(x, y)
    analyzer.plot_importance(save_path=str(PLOT_PATH))

print("save_feature_importance_plot function defined.")

save_feature_importance_plot function defined.


## 3. Data Loading

In [37]:
def build_feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    working = frame.copy()

    gender_encoded = working['Gender'].astype(str).str.lower().map({'female': 0.0, 'male': 1.0}).fillna(0.5)
    age = pd.to_numeric(working['ageAtEntry'], errors='coerce')
    mmse = pd.to_numeric(working['mmse'], errors='coerce')
    cdr = pd.to_numeric(working['cdr'], errors='coerce')
    memory = pd.to_numeric(working['memory'], errors='coerce')
    safe_mmse = mmse.replace(0, np.nan)

    features = pd.DataFrame({
        'age': age,
        'mmse': mmse,
        'cdr': cdr,
        'memory': memory,
        'gender_encoded': gender_encoded,
        'age_x_mmse': age * mmse,
        'age_x_cdr': age * cdr,
        'age_x_memory': age * memory,
        'mmse_x_cdr': mmse * cdr,
        'mmse_x_memory': mmse * memory,
        'cdr_x_memory': cdr * memory,
        'age_sq': age ** 2,
        'mmse_sq': mmse ** 2,
        'cdr_sq': cdr ** 2,
        'memory_sq': memory ** 2,
        'age_minus_mmse': age - mmse,
        'age_plus_memory': age + memory,
        'age_over_mmse': age / (safe_mmse + 1e-3),
    })

    features = features.replace([np.inf, -np.inf], np.nan)
    features = features.fillna(features.median(numeric_only=True))
    return features

print('build_feature_frame function defined.')

build_feature_frame function defined.


In [38]:
frame = load_dataset()

print(f"Dataset shape: {frame.shape}")
print(f"\nColumns: {list(frame.columns)[:5]}... (showing first 5)")
print(f"\nClass distribution:")
print(frame["dx1"].value_counts().sort_index())

Dataset shape: (1229, 7)

Columns: ['Subject', 'Gender', 'mmse', 'ageAtEntry', 'cdr']... (showing first 5)

Class distribution:
dx1
'AD Dementia'           846
'No dementia'            17
'uncertain dementia'    366
Name: count, dtype: int64


In [39]:
from sklearn.preprocessing import LabelEncoder

raw_features = build_feature_frame(frame)
feature_frame = raw_features[ENGINEERED_FEATURE_NAMES].copy()

target_encoder = LabelEncoder()
targets = target_encoder.fit_transform(frame[TARGET_COLUMN].astype(str))
CLASS_NAMES = target_encoder.classes_.tolist()
features = feature_frame

print('Class mapping:')
for index, class_name in enumerate(CLASS_NAMES):
    print(f'{index} -> {class_name}')

print(f'\nFeature shape: {features.shape}')
print(f'Target shape: {targets.shape}')

Class mapping:
0 -> 'AD Dementia'
1 -> 'No dementia'
2 -> 'uncertain dementia'

Feature shape: (1229, 18)
Target shape: (1229,)


## 4. 5-Fold Cross-Validation Training

In [40]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_accuracies: List[float] = []
oof_true: List[int] = []
oof_pred: List[int] = []
best_global_val_acc = -1.0
best_fold_bundle: Optional[Dict] = None
best_fold_val_x: Optional[np.ndarray] = None
best_fold_val_y: Optional[np.ndarray] = None

print(f"Starting {N_SPLITS}-fold cross-validation...\n")

Starting 5-fold cross-validation...



In [41]:
for fold_idx, (train_val_indices, test_indices) in enumerate(skf.split(features, targets), start=1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx}/{N_SPLITS}")
    print(f"{'='*60}")

    # Get train/validation data from this fold
    fold_train_features = features.iloc[train_val_indices].reset_index(drop=True)
    fold_train_targets = targets[train_val_indices]

    # Split fold into train/val
    fold_train_indices, fold_val_indices = train_test_split(
        np.arange(len(fold_train_features)),
        test_size=0.2,
        stratify=fold_train_targets,
        random_state=SEED,
    )

    train_frame = fold_train_features.iloc[fold_train_indices].reset_index(drop=True)
    val_frame = fold_train_features.iloc[fold_val_indices].reset_index(drop=True)
    y_train = fold_train_targets[fold_train_indices]
    y_val = fold_train_targets[fold_val_indices]

    print(f'Train size: {len(train_frame)}, Val size: {len(val_frame)}')

    # Preprocessing
    preprocessor = ClinicalTabularPreprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
    x_train = preprocessor.fit_transform(train_frame)
    x_val = preprocessor.transform(val_frame)

    # SMOTE on training data only
    x_train_resampled, y_train_resampled = make_smote(x_train, y_train)
    print(f'After SMOTE: Train size = {len(x_train_resampled)}')

    # Data loaders
    train_loader = make_loaders(x_train_resampled, y_train_resampled, shuffle=True)
    val_loader = make_loaders(x_val, y_val, shuffle=False)

    # Model setup
    model = AlzheimerDNN(input_features=x_train.shape[1], num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)

    best_fold_acc = -1.0
    best_fold_state = None

    # Training loop for this fold
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        if val_acc > best_fold_acc:
            best_fold_acc = val_acc
            best_fold_state = copy.deepcopy(model.state_dict())

        if epoch % 10 == 0 or epoch == 1:
            print(
                f'  Epoch {epoch:03d}/{EPOCHS} | '
                f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
                f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}'
            )

    # Load best model for this fold
    if best_fold_state is None:
        raise RuntimeError(f'Fold {fold_idx} did not produce a valid checkpoint.')

    model.load_state_dict(best_fold_state)
    val_loss, val_acc, y_true, y_pred, y_prob = evaluate(model, val_loader, criterion, device)

    fold_accuracies.append(val_acc)
    oof_true.extend(y_true.tolist())
    oof_pred.extend(y_pred.tolist())

    print(f'\nFold {fold_idx} Best Validation Accuracy: {best_fold_acc:.4f}')

    # Track best fold globally
    if val_acc > best_global_val_acc:
        best_global_val_acc = val_acc
        best_fold_bundle = {
            'fold': fold_idx,
            'model_state_dict': copy.deepcopy(best_fold_state),
            'preprocessor': copy.deepcopy(preprocessor),
            'feature_names': FEATURE_NAMES,
            'class_names': CLASS_NAMES,
            'val_accuracy': val_acc,
            'val_loss': val_loss,
        }
        best_fold_val_x = val_frame[ENGINEERED_FEATURE_NAMES].to_numpy(dtype=np.float32)
        best_fold_val_y = y_val.copy()

print(f"\n{'='*60}")
print('Cross-validation completed.')
print(f"{'='*60}")


FOLD 1/5
Train size: 786, Val size: 197
After SMOTE: Train size = 1623
  Epoch 001/50 | Train Loss: 2.6481 | Train Acc: 0.6827 | Val Loss: 1.3368 | Val Acc: 0.7563
  Epoch 010/50 | Train Loss: 0.8953 | Train Acc: 0.8071 | Val Loss: 0.8200 | Val Acc: 0.7563
  Epoch 020/50 | Train Loss: 0.8081 | Train Acc: 0.8195 | Val Loss: 0.8256 | Val Acc: 0.7513
  Epoch 030/50 | Train Loss: 0.7780 | Train Acc: 0.8281 | Val Loss: 0.8274 | Val Acc: 0.7614
  Epoch 040/50 | Train Loss: 0.7671 | Train Acc: 0.8361 | Val Loss: 0.8066 | Val Acc: 0.7716
  Epoch 050/50 | Train Loss: 0.7168 | Train Acc: 0.8472 | Val Loss: 0.7935 | Val Acc: 0.7614

Fold 1 Best Validation Accuracy: 0.7716

FOLD 2/5
Train size: 786, Val size: 197
After SMOTE: Train size = 1623
  Epoch 001/50 | Train Loss: 2.4840 | Train Acc: 0.6802 | Val Loss: 1.5265 | Val Acc: 0.6904
  Epoch 010/50 | Train Loss: 0.9109 | Train Acc: 0.8016 | Val Loss: 0.8655 | Val Acc: 0.7005
  Epoch 020/50 | Train Loss: 0.8290 | Train Acc: 0.8207 | Val Loss: 0.8

## 5. Cross-Validation Results

In [42]:
mean_accuracy = float(np.mean(fold_accuracies))
std_accuracy = float(np.std(fold_accuracies))
per_class_f1 = f1_score(np.array(oof_true), np.array(oof_pred), average=None, labels=list(range(NUM_CLASSES)), zero_division=0)

print(f"\n{'='*60}")
print('CROSS-VALIDATION SUMMARY')
print(f"{'='*60}")
print(f'Mean Accuracy across {N_SPLITS} folds: {mean_accuracy:.4f} ± {std_accuracy:.4f}')
print('\nPer-fold Accuracies:')
for i, acc in enumerate(fold_accuracies, 1):
    print(f'  Fold {i}: {acc:.4f}')

print('\nPer-class F1 Scores:')
for class_name, score in zip(CLASS_NAMES, per_class_f1):
    print(f'  {class_name}: {score:.4f}')


CROSS-VALIDATION SUMMARY
Mean Accuracy across 5 folds: 0.7675 ± 0.0134

Per-fold Accuracies:
  Fold 1: 0.7716
  Fold 2: 0.7614
  Fold 3: 0.7868
  Fold 4: 0.7462
  Fold 5: 0.7716

Per-class F1 Scores:
  'AD Dementia': 0.8021
  'No dementia': 1.0000
  'uncertain dementia': 0.7090


## 6. Save Best Model Checkpoint

In [43]:
if best_fold_bundle is None or best_fold_val_x is None or best_fold_val_y is None:
    raise RuntimeError("No best fold checkpoint was produced.")

save_checkpoint(best_fold_bundle)
print(f"✓ Best model checkpoint saved from Fold {best_fold_bundle['fold']}")
print(f"  Path: {SAVE_PATH}")
print(f"  Validation Accuracy: {best_fold_bundle['val_accuracy']:.4f}")

✓ Best model checkpoint saved from Fold 3
  Path: models\saved\dnn_best.pth
  Validation Accuracy: 0.7868


## 7. Feature Importance Analysis

In [44]:
# Load best model
best_model = AlzheimerDNN(input_features=len(FEATURE_NAMES), num_classes=NUM_CLASSES).to(device)
best_model.load_state_dict(best_fold_bundle["model_state_dict"])
best_model.eval()

print("Best model loaded.")

Best model loaded.


In [45]:
# Compute and plot feature importance
class _PreprocessorAdapter:
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.feature_order = getattr(preprocessor, 'feature_order', FEATURE_NAMES)

    def to_torch(self, x: np.ndarray, device=None):
        if hasattr(self.preprocessor, 'to_torch'):
            try:
                return self.preprocessor.to_torch(x, device=device)
            except Exception:
                pass
        frame = pd.DataFrame(x, columns=self.feature_order)
        transformed = self.preprocessor.transform(frame)
        tensor = torch.tensor(transformed, dtype=torch.float32)
        return tensor.to(device) if device is not None else tensor

analyzer = FeatureImportanceAnalyzer(
    model=best_model,
    preprocessing_pipeline=_PreprocessorAdapter(best_fold_bundle["preprocessor"]),
    feature_names=FEATURE_NAMES,
    device=device,
)

save_feature_importance_plot(analyzer, best_fold_val_x, best_fold_val_y)
print(f"✓ Feature importance plot saved to {PLOT_PATH}")

✓ Feature importance plot saved to evaluation\plots\dnn_feature_importance.png


## 8. Final Summary

In [46]:
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
print(f"Number of folds: {N_SPLITS}")
print(f"Epochs per fold: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"\nCross-validation Metrics:")
print(f"  Mean Accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
print(f"  Best Fold: {best_fold_bundle['fold']} (Acc: {best_fold_bundle['val_accuracy']:.4f})")
print(f"\nOutputs saved:")
print(f"  Model: {SAVE_PATH}")
print(f"  Feature Importance: {PLOT_PATH}")
print(f"\n✓ Training Complete!")


TRAINING SUMMARY
Number of folds: 5
Epochs per fold: 50
Batch size: 32

Cross-validation Metrics:
  Mean Accuracy: 0.7675 ± 0.0134
  Best Fold: 3 (Acc: 0.7868)

Outputs saved:
  Model: models\saved\dnn_best.pth
  Feature Importance: evaluation\plots\dnn_feature_importance.png

✓ Training Complete!
